In [0]:
import os
import sys

import mlflow

sys.path.insert(0, os.path.abspath("..")) # este notebook vive en src/notebooks; el módulo, en src/modelos
from modelos import pronostico as P

DESDE = "2022-01-02" # SE 1 de 2022: la primera emisión del backtest
CADA = 4  # semanas entre reentrenamientos; con 1, reentrena cada semana (y tarda 4 veces más)
usuario = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{usuario}/vigia-dengue-modelo")

In [0]:
features = spark.table("vigia.gold.features_provincia_semana").toPandas()
canal = spark.table("vigia.gold.canal_endemico").toPandas()
semanas = (spark.table("vigia.gold.dim_calendario_epi")
           .select("anio_epi", "semana_epi", "inicio_semana").distinct().toPandas())

f = P.preparar(features)
base = P.evaluacion(f, canal, semanas, DESDE)
print(f"{len(f)} filas de features; {base['emision'].nunique()} emisiones en el backtest; "
      f"{base['evaluable'].mean():.0%} de sus filas tienen valor real y canal")

In [0]:
lineas = P.lineas_base(base)

def metricas(modelo, partes):
    """El error del backtest de un modelo, para MLflow"""
    error = P.error_por_horizonte(P.unir(base, partes)).set_index("modelo")
    return error.loc[modelo, ["mae_h1", "mae_h2", "mae_h3", "mae_h4", "mae_medio", "mae_relativo"]].to_dict()


for modelo in ("canal", "ingenuo"):
    with mlflow.start_run(run_name=f"backtest_{modelo}"):
        mlflow.log_params({"modelo": modelo, "desde": DESDE})
        mlflow.log_metrics(metricas(modelo, [lineas]))

variantes = {"gbm_global": P.VARIABLES,
             "gbm_sin_clima": [v for v in P.VARIABLES if v not in P.VARIABLES_CLIMA]}
modelos = []
for modelo, variables in variantes.items():
    with mlflow.start_run(run_name=f"backtest_{modelo}"):
        pronosticos = P.backtest(f, DESDE, variables=variables, cada=CADA, modelo=modelo)
        mlflow.log_params({"modelo": modelo, "desde": DESDE, "reentrenar_cada": CADA,
                           "semanas_etiqueta": P.SEMANAS_ETIQUETA, "variables": len(variables),
                           **P.HIPERPARAMETROS})
        mlflow.log_text("\n".join(variables), "variables.txt")
        mlflow.log_metrics(metricas(modelo, [lineas, pronosticos]))
    modelos.append(pronosticos)
    print(f"{modelo}: {len(pronosticos)} pronósticos")

In [0]:
pronosticos = P.unir(base, [lineas, *modelos])
(spark.createDataFrame(P.para_spark(pronosticos)).write.mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable("vigia.ml.backtest_pronosticos"))
display(P.error_por_horizonte(pronosticos).round(2))